In [1]:
import json
from pathlib import Path
from gossiplearning.history import History
from gossiplearning.config import Config
from utils.evaluation import evaluate_simulations
from utils.metrics import SimulationMetrics, average_metrics, dump_experiment_metrics
import pandas as pd

2025-03-13 10:50:35.209944: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-13 10:50:35.232029: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-13 10:50:35.372683: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-13 10:50:35.374184: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-13 10:50:36.162169: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

In [2]:
dataset_base_dir = Path(f"data/datasets/porto_10n_3k")
network_dir = Path(f"data/networks/porto_10n_3k")
ds_name = "4in_notscaled"
evaluate_generalization = True
evaluate_drop_antenna = False
network_variations = 10
failure_probabilites = [0.0025, 0.005, 0.01]
results_path = Path("experiments/node_failures")

In [ ]:
for failure_probabilty in failure_probabilites:
        exp = f"experiments/node_failures/{failure_probabilty}"
        with open(f"{exp}/0/config.json", "r") as f:
            config = json.load(f)

        config = Config.model_validate(config)
        config.workspace_dir = Path(exp)

        aggregated_plots = Path(config.workspace_dir) / "plots"
        aggregated_plots.mkdir(exist_ok=True, parents=True)

        (
            sim_node_metrics,
            drop_antenna_metrics,
        ) = evaluate_simulations(
            n_sim=network_variations,
            config=config,
            dataset_base_dir=dataset_base_dir,
            evaluate_generalization=evaluate_generalization,
            eval_drop_tower=evaluate_drop_antenna,
            start=0,
            ds_name=ds_name,
            network_dir=network_dir,
        )

        gossip_metrics = [m for sm in sim_node_metrics for m in sm.gossip]
        single_metrics = [m for sm in sim_node_metrics for m in sm.single]
        centralized_metrics = [m for sm in sim_node_metrics for m in sm.centralized]
        gossip_gen_metrics = [
            m for sm in sim_node_metrics for m in sm.gossip_generalized
        ]
        single_gen_metrics = [
            m for sm in sim_node_metrics for m in sm.single_generalized
        ]
        centralized_gen_metrics = [
            m for sm in sim_node_metrics for m in sm.centralized_generalized
        ]

        averaged_metrics = SimulationMetrics(
            gossip=average_metrics(gossip_metrics),
            single_training=average_metrics(single_metrics),
            centralized=average_metrics(centralized_metrics),
        )

        dump_experiment_metrics(
            gossip_metrics=gossip_metrics,
            single_metrics=single_metrics,
            centralized=centralized_metrics,
            file=Path(config.workspace_dir) / "metrics.csv",
        )

        if evaluate_generalization:
            averaged_generalization_metrics = SimulationMetrics(
                gossip=average_metrics(gossip_gen_metrics),
                single_training=average_metrics(single_gen_metrics),
                centralized=average_metrics(centralized_gen_metrics),
            )

            dump_experiment_metrics(
                gossip_metrics=gossip_gen_metrics,
                single_metrics=single_gen_metrics,
                centralized=centralized_gen_metrics,
                file=Path(config.workspace_dir) / "metrics_generalized.csv",
            )

In [26]:
data = {}
for failure_probabilty in failure_probabilites:
    data[failure_probabilty] = {}
    total_failures = 0

    values = {
        'Classical MSE': [],
        'Not Failed Nodes Classical MSE': [],
        'Failed Nodes Classical MSE': [],
        'Generalized MSE': [],
        'Not Failed Nodes Generalized MSE':  [],
        'Failed Nodes Generalized MSE': [],
    }

    run_with_failures = network_variations
    for i in range(network_variations):
        with open(results_path / f"{failure_probabilty}/{i}/history.json") as f:
            history_dict = json.load(f)
        history = History(**history_dict)
        failed_nodes = list(history.nodes_failures_history.keys())

        if failed_nodes == []:
            run_with_failures -= 1

        for failures in history.nodes_failures_history.values():
            total_failures += len(failures)

        metrics = pd.read_csv(results_path / f"{failure_probabilty}/{i}/metrics.csv")
        metrics.columns.values[0] = 'id'
        metrics = metrics[metrics['id'].str.contains(('^gossip\s\d+'))]
        metrics['id'] = metrics['id'].str.replace(r'gossip\s', '', regex=True)
        not_failed_node_metrics = metrics[~metrics['id'].isin(failed_nodes)]
        failed_node_metrics = metrics[metrics['id'].isin(failed_nodes)]

        generalization_metrics = pd.read_csv(results_path / f"{failure_probabilty}/{i}/generalization_metrics.csv")
        generalization_metrics.columns.values[0] = 'id'
        generalization_metrics = generalization_metrics[generalization_metrics['id'].str.contains(('^gossip\s\d+'))]
        generalization_metrics['id'] = generalization_metrics['id'].str.replace(r'gossip\s', '', regex=True)
        not_failed_node_generalization_metrics = generalization_metrics[~generalization_metrics['id'].isin(failed_nodes)]
        failed_node_generalization_metrics = generalization_metrics[generalization_metrics['id'].isin(failed_nodes)]


        values['Classical MSE'].append(metrics['mse'].mean())
        values['Not Failed Nodes Classical MSE'].append(not_failed_node_metrics['mse'].mean())
        if not failed_node_metrics.empty:
            values['Failed Nodes Classical MSE'].append(failed_node_metrics['mse'].mean())

        values['Generalized MSE'].append(generalization_metrics['mse'].mean())
        values['Not Failed Nodes Generalized MSE'].append(not_failed_node_generalization_metrics['mse'].mean())
        if not failed_node_metrics.empty:
            values['Failed Nodes Generalized MSE'].append(failed_node_generalization_metrics['mse'].mean())

    data[failure_probabilty]['Run with Failures'] = f"{run_with_failures} out of {network_variations}"
    data[failure_probabilty]['Total Failures'] = total_failures
    data[failure_probabilty]['Average Failures per Run'] = total_failures / network_variations

    data[failure_probabilty]['Classical MMSE'] = sum(values['Classical MSE']) / network_variations
    data[failure_probabilty]['Classical STD'] = pd.Series(values['Classical MSE']).std()
    data[failure_probabilty]['Classical RSD'] = pd.Series(values['Classical MSE']).std() / data[failure_probabilty]['Classical MMSE']

    data[failure_probabilty]['Not Failed Nodes Classical MMSE'] = sum(values['Not Failed Nodes Classical MSE']) / network_variations
    data[failure_probabilty]['Not Failed Nodes Classical STD'] = pd.Series(values['Not Failed Nodes Classical MSE']).std()
    data[failure_probabilty]['Not Failed Nodes Classical RSD'] = pd.Series(values['Not Failed Nodes Classical MSE']).std() / data[failure_probabilty]['Not Failed Nodes Classical MMSE']

    data[failure_probabilty]['Failed Nodes Classical MMSE'] = sum(values['Failed Nodes Classical MSE']) / run_with_failures
    data[failure_probabilty]['Failed Nodes Classical STD'] = pd.Series(values['Failed Nodes Classical MSE']).std()
    data[failure_probabilty]['Failed Nodes Classical RSD'] = pd.Series(values['Failed Nodes Classical MSE']).std() / data[failure_probabilty]['Failed Nodes Classical MMSE']

    data[failure_probabilty]['Generalized MMSE'] = sum(values['Generalized MSE']) / network_variations
    data[failure_probabilty]['Generalized STD'] = pd.Series(values['Generalized MSE']).std()
    data[failure_probabilty]['Generalized RSD'] = pd.Series(values['Generalized MSE']).std() / data[failure_probabilty]['Generalized MMSE']

    data[failure_probabilty]['Not Failed Nodes Generalized MMSE'] = sum(values['Not Failed Nodes Generalized MSE']) / network_variations
    data[failure_probabilty]['Not Failed Nodes Generalized STD'] = pd.Series(values['Not Failed Nodes Generalized MSE']).std()
    data[failure_probabilty]['Not Failed Nodes Generalized RSD'] = pd.Series(values['Not Failed Nodes Generalized MSE']).std() / data[failure_probabilty]['Not Failed Nodes Generalized MMSE']

    data[failure_probabilty]['Failed Nodes Generalized MMSE'] = sum(values['Failed Nodes Generalized MSE']) / run_with_failures
    data[failure_probabilty]['Failed Nodes Generalized STD'] = pd.Series(values['Failed Nodes Generalized MSE']).std()
    data[failure_probabilty]['Failed Nodes Generalized RSD'] = pd.Series(values['Failed Nodes Generalized MSE']).std() / data[failure_probabilty]['Failed Nodes Generalized MMSE']

df = pd.DataFrame(data)
print(df.to_latex())




\begin{tabular}{llll}
\toprule
 & 0.002500 & 0.005000 & 0.010000 \\
\midrule
Run with Failures & 6 out of 10 & 9 out of 10 & 10 out of 10 \\
Total Failures & 10 & 18 & 53 \\
Average Failures per Run & 1.000000 & 1.800000 & 5.300000 \\
Classical MMSE & 1306.293782 & 1306.506501 & 1312.109630 \\
Classical STD & 90.129087 & 87.301984 & 90.368602 \\
Classical RSD & 0.068996 & 0.066821 & 0.068873 \\
Not Failed Nodes Classical MMSE & 1277.143267 & 1265.399459 & 1522.627273 \\
Not Failed Nodes Classical STD & 205.679116 & 313.597885 & 553.540594 \\
Not Failed Nodes Classical RSD & 0.161046 & 0.247825 & 0.363543 \\
Failed Nodes Classical MMSE & 1593.411487 & 1169.234144 & 1195.881839 \\
Failed Nodes Classical STD & 1875.999739 & 646.379343 & 267.649956 \\
Failed Nodes Classical RSD & 1.177348 & 0.552823 & 0.223810 \\
Generalized MMSE & 2958.399789 & 3150.170887 & 3227.145988 \\
Generalized STD & 962.095565 & 1073.948180 & 1170.181624 \\
Generalized RSD & 0.325208 & 0.340917 & 0.362606 \\
Not F